In [158]:
import pandas as pd
import numpy as np
import re

In [159]:
df = pd.read_csv("data/books_raw.csv")


In [160]:
def clean_price(price_str):
    price_num = re.sub(r"[^\d.]", "", str(price_str))
    return float(price_num) if price_num else np.nan

rating_map = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}

df['Price_clean'] = df['Price'].apply(clean_price)
df['Rating_clean'] = df['Rating'].str.strip().map(rating_map)

df.head()

,Title,Price,Availability,Rating,Category,Price_clean,Rating_clean
0,A Light in the Attic,Â£51.77,In stock,Three,Poetry,51.77,3
1,Tipping the Velvet,Â£53.74,In stock,One,Historical Fiction,53.74,1
2,Soumission,Â£50.10,In stock,One,Fiction,50.10,1
3,Sharp Objects,Â£47.82,In stock,Four,Mystery,47.82,4
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,History,54.23,5


In [161]:
# Normalize text
df['Title_clean'] = df['Title'].str.lower().str.strip()
df['Category_clean'] = df['Category'].str.lower().str.strip()
df.head()

,Title,Price,Availability,Rating,Category,Price_clean,Rating_clean,Title_clean,Category_clean
0,A Light in the Attic,Â£51.77,In stock,Three,Poetry,51.77,3,a light in the attic,poetry
1,Tipping the Velvet,Â£53.74,In stock,One,Historical Fiction,53.74,1,tipping the velvet,historical fiction
2,Soumission,Â£50.10,In stock,One,Fiction,50.10,1,soumission,fiction
3,Sharp Objects,Â£47.82,In stock,Four,Mystery,47.82,4,sharp objects,mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,History,54.23,5,sapiens: a brief history of humankind,history


In [162]:
# Convert Availability to numeric
df['Availability_clean'] = df['Availability'].str.lower().str.contains('in stock').astype(int)
df[['Availability', 'Availability_clean']].head()

,Availability,Availability_clean
0,In stock,1
1,In stock,1
2,In stock,1
3,In stock,1
4,In stock,1


In [163]:
#Extract hashtags, mentions, keywords etc
df["Hashtags"] = df["Title_clean"].apply(lambda x: re.findall(r"#(\w+)", str(x)))
df["Mentions"] = df["Title_clean"].apply(lambda x: re.findall(r"@(\w+)", str(x)))

stopwords = {"the","and","of","in","a","to","for","on"}
df["Keywords"] = df["Title_clean"].apply(
    lambda x: [w for w in str(x).split() if w.lower() not in stopwords and len(w) > 2]
)

df =df
df.head()

,Title,Price,Availability,Rating,Category,Price_clean,Rating_clean,Title_clean,Category_clean,Availability_clean,Hashtags,Mentions,Keywords
0,A Light in the Attic,Â£51.77,In stock,Three,Poetry,51.77,3,a light in the attic,poetry,1,[],[],"[light, attic]"
1,Tipping the Velvet,Â£53.74,In stock,One,Historical Fiction,53.74,1,tipping the velvet,historical fiction,1,[],[],"[tipping, velvet]"
2,Soumission,Â£50.10,In stock,One,Fiction,50.10,1,soumission,fiction,1,[],[],[soumission]
3,Sharp Objects,Â£47.82,In stock,Four,Mystery,47.82,4,sharp objects,mystery,1,[],[],"[sharp, objects]"
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,History,54.23,5,sapiens: a brief history of humankind,history,1,[],[],"[sapiens:, brief, history, humankind]"


In [164]:
# Handle missing values & duplicates
print("Total rows before cleaning:", len(df))
df = df.drop_duplicates(subset=['Title_clean'])

# Ensure numeric conversion
df['Price_clean'] = pd.to_numeric(df['Price_clean'], errors='coerce')
df['Rating_clean'] = pd.to_numeric(df['Rating_clean'], errors='coerce')

# Fill missing values safely
df['Price_clean'] = df['Price_clean'].fillna(df['Price_clean'].median())
df['Rating_clean'] = df['Rating_clean'].fillna(df['Rating_clean'].median())
df['Category_clean'] = df['Category_clean'].fillna("unknown")

print("Total rows after cleaning:", len(df))

Total rows before cleaning: 1000
Total rows after cleaning: 999


In [165]:
# Fill missing titles

df["Title_clean"] = df["Title_clean"].fillna("unknown_title")
df.head()

,Title,Price,Availability,Rating,Category,Price_clean,Rating_clean,Title_clean,Category_clean,Availability_clean,Hashtags,Mentions,Keywords
0,A Light in the Attic,Â£51.77,In stock,Three,Poetry,51.77,3,a light in the attic,poetry,1,[],[],"[light, attic]"
1,Tipping the Velvet,Â£53.74,In stock,One,Historical Fiction,53.74,1,tipping the velvet,historical fiction,1,[],[],"[tipping, velvet]"
2,Soumission,Â£50.10,In stock,One,Fiction,50.10,1,soumission,fiction,1,[],[],[soumission]
3,Sharp Objects,Â£47.82,In stock,Four,Mystery,47.82,4,sharp objects,mystery,1,[],[],"[sharp, objects]"
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,History,54.23,5,sapiens: a brief history of humankind,history,1,[],[],"[sapiens:, brief, history, humankind]"


In [166]:
# Convert list obj to strings

df = df.astype(str)

# drop duplicates
df = df.drop_duplicates()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999 entries, 0 to 999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Title               999 non-null    object
 1   Price               999 non-null    object
 2   Availability        999 non-null    object
 3   Rating              999 non-null    object
 4   Category            999 non-null    object
 5   Price_clean         999 non-null    object
 6   Rating_clean        999 non-null    object
 7   Title_clean         999 non-null    object
 8   Category_clean      999 non-null    object
 9   Availability_clean  999 non-null    object
 10  Hashtags            999 non-null    object
 11  Mentions            999 non-null    object
 12  Keywords            999 non-null    object
dtypes: object(13)
memory usage: 109.3+ KB


In [167]:
#Cleaned DF

df.head()

,Title,Price,Availability,Rating,Category,Price_clean,Rating_clean,Title_clean,Category_clean,Availability_clean,Hashtags,Mentions,Keywords
0,A Light in the Attic,Â£51.77,In stock,Three,Poetry,51.77,3,a light in the attic,poetry,1,[],[],"['light', 'attic']"
1,Tipping the Velvet,Â£53.74,In stock,One,Historical Fiction,53.74,1,tipping the velvet,historical fiction,1,[],[],"['tipping', 'velvet']"
2,Soumission,Â£50.10,In stock,One,Fiction,50.1,1,soumission,fiction,1,[],[],['soumission']
3,Sharp Objects,Â£47.82,In stock,Four,Mystery,47.82,4,sharp objects,mystery,1,[],[],"['sharp', 'objects']"
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,History,54.23,5,sapiens: a brief history of humankind,history,1,[],[],"['sapiens:', 'brief', 'history', 'humankind']"


In [168]:
df.to_csv("data/cleaned_books.csv", index=False, encoding="utf-8")
print("Cleaned data saved to data/cleaned_books.csv")

Cleaned data saved to data/cleaned_books.csv
